In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb
import lightgbm
from sklearn.linear_model import Ridge,Lasso

sns.set_style('darkgrid')

In [53]:
train_df = pd.read_csv("train.csv", parse_dates=["date"])
original_train_df = train_df.copy()
test_df = pd.read_csv("test.csv", parse_dates=["date"])

In [54]:
gdp_per_capita_df = pd.read_csv("gdp_per_capita.csv")

years =  ["2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020"]
gdp_per_capita_filtered_df = gdp_per_capita_df.loc[gdp_per_capita_df["Country Name"].isin(train_df["country"].unique()), ["Country Name"] + years].set_index("Country Name")
gdp_per_capita_filtered_df["2010_ratio"] = gdp_per_capita_filtered_df["2010"] / gdp_per_capita_filtered_df.sum()["2010"]
for year in years:
    gdp_per_capita_filtered_df[f"{year}_ratio"] = gdp_per_capita_filtered_df[year] / gdp_per_capita_filtered_df.sum()[year]
gdp_per_capita_filtered_ratios_df = gdp_per_capita_filtered_df[[i+"_ratio" for i in years]]
gdp_per_capita_filtered_ratios_df.columns = [int(i) for i in years]
gdp_per_capita_filtered_ratios_df = gdp_per_capita_filtered_ratios_df.unstack().reset_index().rename(columns = {"level_0": "year", 0: "ratio", "Country Name": "country"})
gdp_per_capita_filtered_ratios_df['year'] = pd.to_datetime(gdp_per_capita_filtered_ratios_df['year'], format='%Y')

# For plotting purposes
gdp_per_capita_filtered_ratios_df_2 = gdp_per_capita_filtered_ratios_df.copy()
gdp_per_capita_filtered_ratios_df_2["year"] = pd.to_datetime(gdp_per_capita_filtered_ratios_df_2['year'].astype(str)) + pd.offsets.YearEnd(1)
gdp_per_capita_filtered_ratios_df = pd.concat([gdp_per_capita_filtered_ratios_df, gdp_per_capita_filtered_ratios_df_2]).reset_index()

In [55]:
gdp_per_capita_filtered_ratios_df_2["year"] = gdp_per_capita_filtered_ratios_df_2["year"].dt.year

In [56]:
train_df_imputed = train_df.copy()
missing_value_ids = train_df.loc[train_df["num_sold"].isna(), "id"].values
print(f"Missing values remaining: {train_df_imputed['num_sold'].isna().sum()}")

train_df_imputed["year"] = train_df_imputed["date"].dt.year
for year in train_df_imputed["year"].unique():
    # Impute Time Series 1 (Canada, Discount Stickers, Holographic Goose)
    target_ratio = gdp_per_capita_filtered_ratios_df_2.loc[(gdp_per_capita_filtered_ratios_df_2["year"] == year) & (gdp_per_capita_filtered_ratios_df_2["country"] == "Norway"), "ratio"].values[0] # Using Norway as should have the best precision
    current_raito = gdp_per_capita_filtered_ratios_df_2.loc[(gdp_per_capita_filtered_ratios_df_2["year"] == year) & (gdp_per_capita_filtered_ratios_df_2["country"] == "Canada"), "ratio"].values[0]
    ratio_can = current_raito / target_ratio
    train_df_imputed.loc[(train_df_imputed["country"] == "Canada") & (train_df_imputed["store"] == "Discount Stickers") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year), "num_sold"] = (train_df_imputed.loc[(train_df_imputed["country"] == "Norway") & (train_df_imputed["store"] == "Discount Stickers") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year), "num_sold"] * ratio_can).values
    
    # Impute Time Series 2 (Only Missing Values)
    current_ts =  train_df_imputed.loc[(train_df_imputed["country"] == "Canada") & (train_df_imputed["store"] == "Premium Sticker Mart") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year)]
    missing_ts_dates = current_ts.loc[current_ts["num_sold"].isna(), "date"]
    train_df_imputed.loc[(train_df_imputed["country"] == "Canada") & (train_df_imputed["store"] == "Premium Sticker Mart") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] = (train_df_imputed.loc[(train_df_imputed["country"] == "Norway") & (train_df_imputed["store"] == "Premium Sticker Mart") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] * ratio_can).values

    # Impute Time Series 3 (Only Missing Values)
    current_ts =  train_df_imputed.loc[(train_df_imputed["country"] == "Canada") & (train_df_imputed["store"] == "Stickers for Less") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year)]
    missing_ts_dates = current_ts.loc[current_ts["num_sold"].isna(), "date"]
    train_df_imputed.loc[(train_df_imputed["country"] == "Canada") & (train_df_imputed["store"] == "Stickers for Less") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] = (train_df_imputed.loc[(train_df_imputed["country"] == "Norway") & (train_df_imputed["store"] == "Stickers for Less") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] * ratio_can).values
    
    # Impute Time Series 4 (Kenya, Discount Stickers, Holographic Goose)
    current_raito = gdp_per_capita_filtered_ratios_df_2.loc[(gdp_per_capita_filtered_ratios_df_2["year"] == year) & (gdp_per_capita_filtered_ratios_df_2["country"] == "Kenya"), "ratio"].values[0]
    ratio_ken = current_raito / target_ratio
    train_df_imputed.loc[(train_df_imputed["country"] == "Kenya") & (train_df_imputed["store"] == "Discount Stickers") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year), "num_sold"] = (train_df_imputed.loc[(train_df_imputed["country"] == "Norway") & (train_df_imputed["store"] == "Discount Stickers") & (train_df_imputed["product"] == "Holographic Goose")& (train_df_imputed["year"] == year), "num_sold"] * ratio_ken).values

    # Impute Time Series 5 (Only Missing Values)
    current_ts = train_df_imputed.loc[(train_df_imputed["country"] == "Kenya") & (train_df_imputed["store"] == "Premium Sticker Mart") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year)]
    missing_ts_dates = current_ts.loc[current_ts["num_sold"].isna(), "date"]
    train_df_imputed.loc[(train_df_imputed["country"] == "Kenya") & (train_df_imputed["store"] == "Premium Sticker Mart") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] = (train_df_imputed.loc[(train_df_imputed["country"] == "Norway") & (train_df_imputed["store"] == "Premium Sticker Mart") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] * ratio_ken).values

    # Impute Time Series 6 (Only Missing Values)
    current_ts = train_df_imputed.loc[(train_df_imputed["country"] == "Kenya") & (train_df_imputed["store"] == "Stickers for Less") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year)]
    missing_ts_dates = current_ts.loc[current_ts["num_sold"].isna(), "date"]
    train_df_imputed.loc[(train_df_imputed["country"] == "Kenya") & (train_df_imputed["store"] == "Stickers for Less") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] = (train_df_imputed.loc[(train_df_imputed["country"] == "Norway") & (train_df_imputed["store"] == "Stickers for Less") & (train_df_imputed["product"] == "Holographic Goose") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] * ratio_ken).values

    # Impute Time Series 7 (Only Missing Values)
    current_ts = train_df_imputed.loc[(train_df_imputed["country"] == "Kenya") & (train_df_imputed["store"] == "Discount Stickers") & (train_df_imputed["product"] == "Kerneler") & (train_df_imputed["year"] == year)]
    missing_ts_dates = current_ts.loc[current_ts["num_sold"].isna(), "date"]
    train_df_imputed.loc[(train_df_imputed["country"] == "Kenya") & (train_df_imputed["store"] == "Discount Stickers") & (train_df_imputed["product"] == "Kerneler") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] = (train_df_imputed.loc[(train_df_imputed["country"] == "Norway") & (train_df_imputed["store"] == "Discount Stickers") & (train_df_imputed["product"] == "Kerneler") & (train_df_imputed["year"] == year) & (train_df_imputed["date"].isin(missing_ts_dates)), "num_sold"] * ratio_ken).values
    
print(f"Missing values remaining: {train_df_imputed['num_sold'].isna().sum()}")

Missing values remaining: 8871
Missing values remaining: 2


In [57]:
missing_rows = train_df_imputed.loc[train_df_imputed["num_sold"].isna()]
display(missing_rows)
train_df_imputed.loc[train_df_imputed["id"] == 23719, "num_sold"] = 4
train_df_imputed.loc[train_df_imputed["id"] == 207003, "num_sold"] = 195

print(f"Missing values remaining: {train_df_imputed['num_sold'].isna().sum()}")

,id,date,country,store,product,num_sold,year
23719,23719,2010-09-21,Kenya,Discount Stickers,Kerneler Dark Mode,NaN,2010
207003,207003,2016-04-19,Canada,Discount Stickers,Kerneler,NaN,2016


Missing values remaining: 0


In [58]:
store_weights = train_df_imputed.groupby("store")["num_sold"].sum()/train_df_imputed["num_sold"].sum()
store_weights

store
Discount Stickers       0.184716
Premium Sticker Mart    0.441564
Stickers for Less       0.373720
Name: num_sold, dtype: float64

In [59]:
product_df = train_df_imputed.groupby(["date","product"])["num_sold"].sum().reset_index()
product_ratio_df = product_df.pivot(index="date", columns="product", values="num_sold")
product_ratio_df = product_ratio_df.apply(lambda x: x/x.sum(),axis=1)
product_ratio_df = product_ratio_df.stack().rename("ratios").reset_index()
product_ratio_df.head(4)

,date,product,ratios
0,2010-01-01,Holographic Goose,0.052441
1,2010-01-01,Kaggle,0.329305
2,2010-01-01,Kaggle Tiers,0.291165
3,2010-01-01,Kerneler,0.152373


In [60]:
train_df_imputed

,id,date,country,store,product,num_sold,year
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,141.557387,2010
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.000000,2010
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.000000,2010
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.000000,2010
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.000000,2010
...,...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.000000,2016
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.000000,2016
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.000000,2016
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.000000,2016


In [61]:
original_train_df_imputed = train_df_imputed.copy()
train_df_imputed = train_df_imputed.groupby(["date"])["num_sold"].sum().reset_index()

In [62]:
train_df_imputed

,date,num_sold
0,2010-01-01,85710.772635
1,2010-01-02,82699.983533
2,2010-01-03,88476.100742
3,2010-01-04,68206.949986
4,2010-01-05,65832.192575
...,...,...
2552,2016-12-27,64651.885691
2553,2016-12-28,71252.729934
2554,2016-12-29,77548.729112
2555,2016-12-30,82028.751480


In [63]:
test_total_sales_df = test_df.groupby(["date"])["id"].first().reset_index().drop(columns="id")
test_total_sales_dates = test_total_sales_df[["date"]]

In [64]:
def feature_engineer(df):
    new_df = df.copy()
    new_df["month"] = df["date"].dt.month
    new_df["month_sin"] = np.sin(new_df['month'] * (2 * np.pi / 12))
    new_df["month_cos"] = np.cos(new_df['month'] * (2 * np.pi / 12))
    new_df["day_of_week"] = df["date"].dt.dayofweek
    new_df["day_of_week"] = new_df["day_of_week"].apply(lambda x: 0 if x<=3 else(1 if x==4 else (2 if x==5 else (3))))    
    new_df["day_of_year"] = df['date'].apply(
        lambda x: x.timetuple().tm_yday if not (x.is_leap_year and x.month > 2) else x.timetuple().tm_yday - 1
    )
    
    new_df['day_sin'] = np.sin(new_df['day_of_year'] * (2 * np.pi /  365.0))
    new_df['day_cos'] = np.cos(new_df['day_of_year'] * (2 * np.pi /  365.0))    
    new_df["important_dates"] = new_df["day_of_year"].apply(lambda x: x if x in [1,2,3,4,5,6,7,8,9,10,99, 100, 101, 125,126,355,256,357,358,359,360,361,362,363,364,365] else 0)
    
    new_df = new_df.drop(columns=["date","month","day_of_year"])
    new_df = pd.get_dummies(new_df, columns = ["important_dates","day_of_week"], drop_first=True)
    
    return new_df

In [65]:
train_total_sales_df = feature_engineer(train_df_imputed)
test_total_sales_df = feature_engineer(test_total_sales_df)

In [66]:
import holidays
train_df_tmp = train_df.copy()
test_df_tmp = test_df.copy()
alpha2 = dict(zip(np.sort(train_df.country.unique()), ['CA', 'FI', 'IT', 'KE', 'NO', 'SG']))
h = {c: holidays.country_holidays(a, years=range(2010, 2020)) for c, a in alpha2.items()}
train_df_tmp['is_holiday'] = 0
test_df_tmp['is_holiday'] = 0
for c in alpha2:
    train_df_tmp.loc[train_df_tmp.country==c, 'is_holiday'] = train_df_tmp.date.isin(h[c]).astype(int)
    test_df_tmp.loc[test_df_tmp.country==c, 'is_holiday'] = test_df_tmp.date.isin(h[c]).astype(int)


train_total_sales_df['is_holiday'] = (train_df_tmp.groupby(["date"])["is_holiday"].sum().reset_index())["is_holiday"]
test_total_sales_df['is_holiday'] = (test_df_tmp.groupby(["date"])["is_holiday"].sum().reset_index())["is_holiday"]

In [67]:
y = train_total_sales_df["num_sold"]
X = train_total_sales_df.drop(columns="num_sold")

In [68]:
X

,month_sin,month_cos,day_sin,day_cos,important_dates_1,important_dates_2,important_dates_3,important_dates_4,important_dates_5,important_dates_6,...,important_dates_360,important_dates_361,important_dates_362,important_dates_363,important_dates_364,important_dates_365,day_of_week_1,day_of_week_2,day_of_week_3,is_holiday
0,5.000000e-01,0.866025,1.721336e-02,0.999852,True,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,90
1,5.000000e-01,0.866025,3.442161e-02,0.999407,False,True,False,False,False,False,...,False,False,False,False,False,False,False,True,False,0
2,5.000000e-01,0.866025,5.161967e-02,0.998667,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,0
3,5.000000e-01,0.866025,6.880243e-02,0.997630,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,0
4,5.000000e-01,0.866025,8.596480e-02,0.996298,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2552,-2.449294e-16,1.000000,-6.880243e-02,0.997630,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,15
2553,-2.449294e-16,1.000000,-5.161967e-02,0.998667,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,0
2554,-2.449294e-16,1.000000,-3.442161e-02,0.999407,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,0
2555,-2.449294e-16,1.000000,-1.721336e-02,0.999852,False,False,False,False,False,False,...,False,False,False,False,True,False,True,False,False,0


In [69]:
from sklearn.metrics import mean_absolute_percentage_error
model = Lasso(tol=1e-2, max_iter=1000000, random_state=0)
model.fit(X, y)
preds = model.predict(test_total_sales_df)
test_total_sales_dates["num_sold"] = preds
mean_absolute_percentage_error(y,model.predict(X))


0.07015051198859394

In [70]:
product_ratio_2017_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()
product_ratio_2018_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2016].copy()
product_ratio_2019_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()

product_ratio_2017_df["date"] = product_ratio_2017_df["date"] + pd.DateOffset(years=2)
product_ratio_2018_df["date"] = product_ratio_2018_df["date"] + pd.DateOffset(years=2)
product_ratio_2019_df["date"] =  product_ratio_2019_df["date"] + pd.DateOffset(years=4)

forecasted_ratios_df = pd.concat([product_ratio_2017_df, product_ratio_2018_df, product_ratio_2019_df])

In [71]:
store_weights_df = store_weights.reset_index()
test_sub_df = pd.merge(test_df, test_total_sales_dates, how="left", on="date")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"day_num_sold"})
# Adding in the product ratios
test_sub_df = pd.merge(test_sub_df, store_weights_df, how="left", on="store")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"store_ratio"})
# Adding in the country ratios
test_sub_df["year"] = test_sub_df["date"].dt.year
test_sub_df = pd.merge(test_sub_df, gdp_per_capita_filtered_ratios_df_2, how="left", on=["year", "country"])
test_sub_df = test_sub_df.rename(columns = {"ratio":"country_ratio"})
# Adding in the product ratio
test_sub_df = pd.merge(test_sub_df, forecasted_ratios_df, how="left", on=["date", "product"])
test_sub_df = test_sub_df.rename(columns = {"ratios":"product_ratio"})

# Disaggregating the forecast
test_sub_df.loc[test_sub_df['country'] == 'Kenya', 'country_ratio'] -= 0.0007/2

test_sub_df["num_sold"] = test_sub_df["day_num_sold"] * test_sub_df["store_ratio"] * test_sub_df["country_ratio"] * test_sub_df["product_ratio"]
test_sub_df["num_sold"] = test_sub_df["num_sold"].round()
display(test_sub_df.head(2))

,id,date,country,store,product,day_num_sold,store_ratio,year,country_ratio,product_ratio,num_sold
0,230130,2017-01-01,Canada,Discount Stickers,Holographic Goose,90256.101882,0.184716,2017,0.17221,0.053755,154.0
1,230131,2017-01-01,Canada,Discount Stickers,Kaggle,90256.101882,0.184716,2017,0.17221,0.350044,1005.0


In [72]:
submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = test_sub_df["num_sold"]
display(submission.head(2))
submission.to_csv('submissionlr.csv', index = False)

,id,num_sold
0,230130,154.0
1,230131,1005.0


In [73]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3,random_state=95)

In [74]:
from sklearn.metrics import mean_absolute_percentage_error as MAPE
def xgb_objective(trial):
    # Hyperparameter search space
    param = {
        'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
    }

    # Fit model
    model = xgb.XGBRegressor(**param, random_state=95)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    # Predict and evaluate
    preds = model.predict(X_test)
    mape = MAPE(y_test, preds)
    return mape


study_XGB = optuna.create_study(direction='minimize')  # Minimize RMSE
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_XGB.optimize(xgb_objective, n_trials=200, show_progress_bar=True)  # Run for 50 trials or 10 minutes

# Best parameters and score
print("Best Trial:")
print(f"  Value: {study_XGB.best_trial.value:.4f}")
print("  Params: ")
for key, value in study_XGB.best_trial.params.items():
    print(f"    {key}: {value}")


xgb_final = xgb.XGBRegressor(**study_XGB.best_params,random_state=95,verbose=-1)
xgb_final.fit(X,y)
print(MAPE(y,xgb_final.predict(X)))

  0%|          | 0/200 [00:00<?, ?it/s]

Best Trial:
  Value: 0.0696
  Params: 
    booster: gbtree
    learning_rate: 0.021553127713466565
    n_estimators: 123
    max_depth: 3
    min_child_weight: 3.598471496680272
    gamma: 4.422839949985445
    subsample: 0.7924283149432773
    colsample_bytree: 0.5772402651319286
    lambda: 0.12216044674978324
    alpha: 0.5142231212709958
0.07113473922361833


c:\Users\dipty\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [13:08:41] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "verbose" } are not used.

  warnings.warn(smsg, UserWarning)


In [75]:
preds = xgb_final.predict(test_total_sales_df)
test_total_sales_dates["num_sold"] = preds
mean_absolute_percentage_error(y,xgb_final.predict(X))

product_ratio_2017_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()
product_ratio_2018_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2016].copy()
product_ratio_2019_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()

product_ratio_2017_df["date"] = product_ratio_2017_df["date"] + pd.DateOffset(years=2)
product_ratio_2018_df["date"] = product_ratio_2018_df["date"] + pd.DateOffset(years=2)
product_ratio_2019_df["date"] =  product_ratio_2019_df["date"] + pd.DateOffset(years=4)

forecasted_ratios_df = pd.concat([product_ratio_2017_df, product_ratio_2018_df, product_ratio_2019_df])

store_weights_df = store_weights.reset_index()
test_sub_df = pd.merge(test_df, test_total_sales_dates, how="left", on="date")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"day_num_sold"})
# Adding in the product ratios
test_sub_df = pd.merge(test_sub_df, store_weights_df, how="left", on="store")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"store_ratio"})
# Adding in the country ratios
test_sub_df["year"] = test_sub_df["date"].dt.year
test_sub_df = pd.merge(test_sub_df, gdp_per_capita_filtered_ratios_df_2, how="left", on=["year", "country"])
test_sub_df = test_sub_df.rename(columns = {"ratio":"country_ratio"})
# Adding in the product ratio
test_sub_df = pd.merge(test_sub_df, forecasted_ratios_df, how="left", on=["date", "product"])
test_sub_df = test_sub_df.rename(columns = {"ratios":"product_ratio"})

# Disaggregating the forecast
test_sub_df.loc[test_sub_df['country'] == 'Kenya', 'country_ratio'] -= 0.0007/2

test_sub_df["num_sold"] = test_sub_df["day_num_sold"] * test_sub_df["store_ratio"] * test_sub_df["country_ratio"] * test_sub_df["product_ratio"]
test_sub_df["num_sold"] = test_sub_df["num_sold"].round()
display(test_sub_df.head(2))

submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = test_sub_df["num_sold"]
display(submission.head(2))

submission.to_csv('submissionxgb.csv', index = False)

,id,date,country,store,product,day_num_sold,store_ratio,year,country_ratio,product_ratio,num_sold
0,230130,2017-01-01,Canada,Discount Stickers,Holographic Goose,90421.195312,0.184716,2017,0.17221,0.053755,155.0
1,230131,2017-01-01,Canada,Discount Stickers,Kaggle,90421.195312,0.184716,2017,0.17221,0.350044,1007.0


,id,num_sold
0,230130,155.0
1,230131,1007.0


In [76]:
from lightgbm import LGBMRegressor


def lgbm_objective(trial):

    lgbm_params = {
        "n_estimators": 2000,
        "subsample": trial.suggest_float("subsample", 0.3, 0.9),
        "min_child_samples": trial.suggest_int("min_child_samples", 60, 100),
        "max_depth": trial.suggest_int("max_depth", 4, 25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.001, 0.1),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.001, 0.1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0)

    }

    lgbm_model = LGBMRegressor(**lgbm_params, random_state=95, verbose=-1)

    lgbm_model.fit(X_train, y_train)
    y_pred = lgbm_model.predict(X_test)
    return MAPE(y_test, y_pred)

study_LGBM = optuna.create_study(study_name="LGBM_Kaggle", direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_LGBM.optimize(lgbm_objective, n_trials=200, show_progress_bar=True)

print("Best trial:", study_LGBM.best_trial)
print("Best parameters:", study_LGBM.best_params)

lgbm_final = LGBMRegressor(**study_LGBM.best_params,n_estimators= 2000,random_state=95,verbose=-1)
lgbm_final.fit(X, y)
y_pred = lgbm_final.predict(X)
print("MAPE:",mean_absolute_percentage_error(y, y_pred))

  0%|          | 0/200 [00:00<?, ?it/s]

Best trial: FrozenTrial(number=130, state=1, values=[0.07274653081867592], datetime_start=datetime.datetime(2025, 1, 11, 13, 9, 53, 974417), datetime_complete=datetime.datetime(2025, 1, 11, 13, 9, 54, 365733), params={'subsample': 0.4024981491189299, 'min_child_samples': 99, 'max_depth': 4, 'learning_rate': 0.0158431937115686, 'lambda_l1': 0.04410649791875963, 'lambda_l2': 0.02808678305402855, 'colsample_bytree': 0.31077161305777296}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'subsample': FloatDistribution(high=0.9, log=False, low=0.3, step=None), 'min_child_samples': IntDistribution(high=100, log=False, low=60, step=1), 'max_depth': IntDistribution(high=25, log=False, low=4, step=1), 'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'lambda_l1': FloatDistribution(high=0.1, log=False, low=0.001, step=None), 'lambda_l2': FloatDistribution(high=0.1, log=False, low=0.001, step=None), 'colsample_bytree': FloatDistribution(high=1.0, l

In [77]:
preds = lgbm_final.predict(test_total_sales_df)
test_total_sales_dates["num_sold"] = preds
mean_absolute_percentage_error(y,lgbm_final.predict(X))

product_ratio_2017_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()
product_ratio_2018_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2016].copy()
product_ratio_2019_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()

product_ratio_2017_df["date"] = product_ratio_2017_df["date"] + pd.DateOffset(years=2)
product_ratio_2018_df["date"] = product_ratio_2018_df["date"] + pd.DateOffset(years=2)
product_ratio_2019_df["date"] =  product_ratio_2019_df["date"] + pd.DateOffset(years=4)

forecasted_ratios_df = pd.concat([product_ratio_2017_df, product_ratio_2018_df, product_ratio_2019_df])

store_weights_df = store_weights.reset_index()
test_sub_df = pd.merge(test_df, test_total_sales_dates, how="left", on="date")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"day_num_sold"})
# Adding in the product ratios
test_sub_df = pd.merge(test_sub_df, store_weights_df, how="left", on="store")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"store_ratio"})
# Adding in the country ratios
test_sub_df["year"] = test_sub_df["date"].dt.year
test_sub_df = pd.merge(test_sub_df, gdp_per_capita_filtered_ratios_df_2, how="left", on=["year", "country"])
test_sub_df = test_sub_df.rename(columns = {"ratio":"country_ratio"})
# Adding in the product ratio
test_sub_df = pd.merge(test_sub_df, forecasted_ratios_df, how="left", on=["date", "product"])
test_sub_df = test_sub_df.rename(columns = {"ratios":"product_ratio"})

# Disaggregating the forecast
test_sub_df.loc[test_sub_df['country'] == 'Kenya', 'country_ratio'] -= 0.0007/2

test_sub_df["num_sold"] = test_sub_df["day_num_sold"] * test_sub_df["store_ratio"] * test_sub_df["country_ratio"] * test_sub_df["product_ratio"]
test_sub_df["num_sold"] = test_sub_df["num_sold"].round()
display(test_sub_df.head(2))

submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = test_sub_df["num_sold"]
display(submission.head(2))

submission.to_csv('submissionlgbm.csv', index = False)

,id,date,country,store,product,day_num_sold,store_ratio,year,country_ratio,product_ratio,num_sold
0,230130,2017-01-01,Canada,Discount Stickers,Holographic Goose,86803.890168,0.184716,2017,0.17221,0.053755,148.0
1,230131,2017-01-01,Canada,Discount Stickers,Kaggle,86803.890168,0.184716,2017,0.17221,0.350044,967.0


,id,num_sold
0,230130,148.0
1,230131,967.0


In [83]:
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import Ridge
model = Ridge(tol=1e-2, max_iter=1000000, random_state=0)
model.fit(X, y)
preds = model.predict(test_total_sales_df)
test_total_sales_dates["num_sold"] = preds
print(mean_absolute_percentage_error(y,model.predict(X)))

product_ratio_2017_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()
product_ratio_2018_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2016].copy()
product_ratio_2019_df = product_ratio_df.loc[product_ratio_df["date"].dt.year == 2015].copy()

product_ratio_2017_df["date"] = product_ratio_2017_df["date"] + pd.DateOffset(years=2)
product_ratio_2018_df["date"] = product_ratio_2018_df["date"] + pd.DateOffset(years=2)
product_ratio_2019_df["date"] =  product_ratio_2019_df["date"] + pd.DateOffset(years=4)

forecasted_ratios_df = pd.concat([product_ratio_2017_df, product_ratio_2018_df, product_ratio_2019_df])

store_weights_df = store_weights.reset_index()
test_sub_df = pd.merge(test_df, test_total_sales_dates, how="left", on="date")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"day_num_sold"})
# Adding in the product ratios
test_sub_df = pd.merge(test_sub_df, store_weights_df, how="left", on="store")
test_sub_df = test_sub_df.rename(columns = {"num_sold":"store_ratio"})
# Adding in the country ratios
test_sub_df["year"] = test_sub_df["date"].dt.year
test_sub_df = pd.merge(test_sub_df, gdp_per_capita_filtered_ratios_df_2, how="left", on=["year", "country"])
test_sub_df = test_sub_df.rename(columns = {"ratio":"country_ratio"})
# Adding in the product ratio
test_sub_df = pd.merge(test_sub_df, forecasted_ratios_df, how="left", on=["date", "product"])
test_sub_df = test_sub_df.rename(columns = {"ratios":"product_ratio"})

# Disaggregating the forecast
test_sub_df.loc[test_sub_df['country'] == 'Kenya', 'country_ratio'] -= 0.0007/2

test_sub_df["num_sold"] = test_sub_df["day_num_sold"] * test_sub_df["store_ratio"] * test_sub_df["country_ratio"] * test_sub_df["product_ratio"]
test_sub_df["num_sold"] = test_sub_df["num_sold"].round()
display(test_sub_df.head(2))

submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = test_sub_df["num_sold"]
display(submission.head(2))
submission.to_csv('submissionlrr.csv', index = False)

0.07016419184118722


,id,date,country,store,product,day_num_sold,store_ratio,year,country_ratio,product_ratio,num_sold
0,230130,2017-01-01,Canada,Discount Stickers,Holographic Goose,89073.000313,0.184716,2017,0.17221,0.053755,152.0
1,230131,2017-01-01,Canada,Discount Stickers,Kaggle,89073.000313,0.184716,2017,0.17221,0.350044,992.0


,id,num_sold
0,230130,152.0
1,230131,992.0


In [91]:
One,Two,Three,Four,Five=pd.read_csv('submissionxgb.csv'),pd.read_csv('submissionlr.csv'),pd.read_csv('submissionlgbm.csv'),pd.read_csv('submissionlrr.csv'),pd.read_csv('submissionall.csv')
submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = .2*One.num_sold+.15*Two.num_sold+.15*Three.num_sold+.15*Four.num_sold+.35*Five.num_sold
submission.to_csv('submissionfinal.csv', index = False)

In [2]:
import pandas as pd
import numpy as np

In [4]:
'''a=pd.read_csv('submissionxgb.csv') #xgb_new
b=pd.read_csv('submissionlr.csv') #lrl_new
c=pd.read_csv('submissionlrr.csv') #lrr_new
d=pd.read_csv('submissionlgbm.csv') #lgbm_new
e1=pd.read_csv('submissionfinal.csv') #final_new
e2=pd.read_csv('submissionmix1.csv') #new_mix1

e,f,g,h,i=pd.read_csv('submissionallweighted.csv'),pd.read_csv('submissionxgb_old.csv'),pd.read_csv('submissionall.csv'),pd.read_csv('submissionlgbm_old.csv'),pd.read_csv('submissionDL.csv') #DL_baseline_old

scores=[.07022,.08342,.08342,.08432,.07019,.07483,.12514,.12549,.12744] #change scores for e1,e2
dfs=[a,b,c,d,e1,e2,e,f,g]
coefficient=[1-i for i in scores]
total=np.sum(coefficient)
submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = 0*len(submission.num_sold)
for i,v in enumerate(dfs):submission["num_sold"] += v.num_sold*coefficient[i] 
submission["num_sold"]/=total

submission.to_csv('submissionmix2.csv', index = False)'''

In [16]:
'''a=pd.read_csv('submissionxgb.csv') #xgb_new
b=pd.read_csv('submissionlr.csv') #lrl_new
e=pd.read_csv('submissionallweighted.csv') #bias

scores=[.07,.08,.12]
dfs=[a,b,e]

coefficient=[1-i for i in scores]
total=np.sum(coefficient)
submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = 0*len(submission.num_sold)
for i,v in enumerate(dfs):submission["num_sold"] += v.num_sold*coefficient[i] 
submission["num_sold"]/=total

submission.to_csv('submissionmix4.csv', index = False)'''

In [ ]:
import pandas as pd
import numpy as np
a=pd.read_csv('submissionfinal_12.csv') 
b=pd.read_csv('submissionmix4_12.csv') 
e=pd.read_csv('submissionmix4_13.csv') #bias

scores=[.07019,.06822,.06904] #evaluate submission.csv and arrange
print(scores)
dfs=[a,b,e]

coefficient=[1/i for i in scores]
print(coefficient)
total=np.sum(coefficient)
submission = pd.read_csv("sample_submission.csv")
submission["num_sold"] = 0*len(submission.num_sold)
for i,v in enumerate(dfs):submission["num_sold"] += v.num_sold*coefficient[i] 
submission["num_sold"]/=total

submission.to_csv('submissionmix1_14.csv', index = False)

[0.07019, 0.06822, 0.06937]
[14.247043738424276, 14.65845793022574, 14.41545336600836]
